In [ ]:
import numpy as np
import pandas as pd
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, f1_score, matthews_corrcoef, confusion_matrix
from sklearn.utils.class_weight import compute_sample_weight
import xgboost as xgb
# ==========================================
# 1. LOAD & CLEAN TARGET LABELS
# ==========================================
ai4i_2020 = fetch_ucirepo(id=601)
df = pd.concat([ai4i_2020.data.features, ai4i_2020.data.targets], axis=1)

failure_cols = ["TWF", "HDF", "PWF", "OSF", "RNF"]
df["Failure_Mode_Sum"] = df[failure_cols].sum(axis=1)



In [2]:
# Clean documented label anomalies:
# 1. Remove rows where Machine failure == 1 but no specific failure mode is flagged (9 rows)
# 2. Remove rows where Machine failure == 0 but RNF == 1 (18 rows)
# 3. For rare simultaneous failures (24 rows), prioritize the most severe physical mode: OSF > PWF > HDF > TWF > RNF
df_clean = df[
    ~((df["Machine failure"] == 1) & (df["Failure_Mode_Sum"] == 0)) &
    ~((df["Machine failure"] == 0) & (df["RNF"] == 1))
].copy()

priority_order = ["OSF", "PWF", "HDF", "TWF", "RNF"]

def get_clean_target(row):
    if row["Machine failure"] == 0:
        return "No_Failure"
    for mode in priority_order:
        if row[mode] == 1:
            return mode
    return "No_Failure"

df_clean["Target"] = df_clean.apply(get_clean_target, axis=1)

# Drop RNF if desired (since RNF is random noise by definition, ~18 rows total), or keep it.
# Here we keep all 5 failure modes + No_Failure (6 classes total).
print("Cleaned Target Distribution:")
print(df_clean["Target"].value_counts())


Cleaned Target Distribution:
Target
No_Failure    9643
HDF            106
OSF             98
PWF             83
TWF             43
Name: count, dtype: int64


In [3]:

# ==========================================
# 2. DOMAIN-PHYSICS FEATURE ENGINEERING
# ==========================================
def engineer_features(data: pd.DataFrame) -> pd.DataFrame:
    X = data.copy()
    
    # 1. Encode ordinal machine quality Type (L=0, M=1, H=2)
    # Tool wear failure thresholds depend directly on product quality Type
    type_map = {"L": 0, "M": 1, "H": 2}
    X["Type_Encoded"] = X["Type"].map(type_map)
    
    # 2. Temperature Difference (Critical for Heat Dissipation Failure - HDF)
    # HDF occurs when temp difference < 8.6 K and rotational speed < 1380 rpm
    X["Temp_Diff"] = X["Process temperature"] - X["Air temperature"]
    X["Temp_Ratio"] = X["Process temperature"] / X["Air temperature"]
    
    # 3. Mechanical Power in Watts (Critical for Power Failure - PWF)
    # P = Torque (Nm) * Angular Velocity (rad/s), where omega = RPM * (2 * pi / 60)
    angular_velocity = X["Rotational speed"] * (2 * np.pi / 60)
    X["Power_Watts"] = X["Torque"] * angular_velocity
    
    # 4. Overstrain Metric (Critical for Overstrain Failure - OSF)
    # OSF occurs when product of Tool wear (min) * Torque (Nm) exceeds type-specific thresholds
    X["Overstrain_Index"] = X["Tool wear"] * X["Torque"]
    
    # 5. Interaction & Stress Ratios
    X["Wear_Per_RPM"] = X["Tool wear"] / (X["Rotational speed"] + 1e-5)
    X["Torque_Per_RPM"] = X["Torque"] / (X["Rotational speed"] + 1e-5)
    X["Thermal_Stress"] = X["Temp_Diff"] * X["Power_Watts"]
    
    # Drop original non-feature columns
    drop_cols = ["Type", "Machine failure", "Failure_Mode_Sum", "Target"] + failure_cols
    feature_cols = [c for c in X.columns if c not in drop_cols]
    
    return X[feature_cols]

X = engineer_features(df_clean)
le = LabelEncoder()
y = le.fit_transform(df_clean["Target"])

print(f"\nEngineered Feature Matrix Shape: {X.shape}")
print("Features:", list(X.columns))



Engineered Feature Matrix Shape: (9973, 13)
Features: ['Air temperature', 'Process temperature', 'Rotational speed', 'Torque', 'Tool wear', 'Type_Encoded', 'Temp_Diff', 'Temp_Ratio', 'Power_Watts', 'Overstrain_Index', 'Wear_Per_RPM', 'Torque_Per_RPM', 'Thermal_Stress']


In [4]:

# ==========================================
# 3. TRAIN / TEST SPLIT (STRATIFIED)
# ==========================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Compute balanced sample weights to counteract severe class imbalance
sample_weights_train = compute_sample_weight(class_weight="balanced", y=y_train)


In [7]:

# ==========================================
# 4. BASELINE XGBOOST (MULTI-CLASS)
# ==========================================
print("\n" + "="*50)
print("TRAINING BASELINE XGBOOST")
print("="*50)

xgb_model = xgb.XGBClassifier(
    objective="multi:softprob",
    num_class=len(le.classes_),
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(X_train, y_train, sample_weight=sample_weights_train)
y_pred_xgb = xgb_model.predict(X_test)

print(f"XGBoost Macro F1: {f1_score(y_test, y_pred_xgb, average='macro'):.4f}")
print(f"XGBoost MCC:      {matthews_corrcoef(y_test, y_pred_xgb):.4f}\n")
print(classification_report(y_test, y_pred_xgb, target_names=le.classes_, digits=4, zero_division=0))



TRAINING BASELINE XGBOOST
XGBoost Macro F1: 0.7897
XGBoost MCC:      0.8261

              precision    recall  f1-score   support

         HDF     0.8750    1.0000    0.9333        21
  No_Failure     0.9958    0.9917    0.9938      1929
         OSF     0.9500    0.9500    0.9500        20
         PWF     0.9444    1.0000    0.9714        17
         TWF     0.0833    0.1250    0.1000         8

    accuracy                         0.9880      1995
   macro avg     0.7697    0.8133    0.7897      1995
weighted avg     0.9900    0.9880    0.9889      1995



In [8]:

# ==========================================
# 6. 5-FOLD STRATIFIED CROSS-VALIDATION CHECK
# ==========================================
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_f1_scores = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    X_tr, y_tr = X.iloc[train_idx], y[train_idx]
    X_va, y_va = X.iloc[val_idx], y[val_idx]
    
    w_tr = compute_sample_weight(class_weight="balanced", y=y_tr)
    
    model = xgb.XGBClassifier(
        objective="multi:softprob",
        num_class=len(le.classes_),
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        random_state=42,
        n_jobs=-1
    )
    model.fit(X_tr, y_tr, sample_weight=w_tr)
    preds = model.predict(X_va)
    cv_f1_scores.append(f1_score(y_va, preds, average="macro"))

print(f"\n5-Fold Stratified CV Macro F1 (XGBoost): {np.mean(cv_f1_scores):.4f} +/- {np.std(cv_f1_scores):.4f}")


5-Fold Stratified CV Macro F1 (XGBoost): 0.8023 +/- 0.0134


# Baseline XGBoost Model Evaluation Analysis (AI4I 2020)

## 1. Overall Performance Summary
* **Accuracy (`98.80%`) & Weighted F1 (`0.9889`):** Heavily dominated by the majority class (`No_Failure`, 1,929 of 1,995 test samples). These metrics are not reliable indicators of fault-detection quality.
* **Matthews Correlation Coefficient (`MCC = 0.8261`):** Strong overall correlation between predicted and actual labels across all classes, accounting for the severe class imbalance.
* **Macro F1 (`0.7897` Test | `0.8023 ± 0.0134` 5-Fold CV):** Consistent generalization between the holdout split and cross-validation (low variance across folds). However, the macro average is pulled down almost entirely by a single failure mode (`TWF`).

---

## 2. Per-Class Diagnostic Breakdown

| Class | Support | Precision | Recall | F1-Score | Diagnostic Status |
| :--- | :---: | :---: | :---: | :---: | :--- |
| **No_Failure** | 1,929 | 0.9958 | 0.9917 | 0.9938 | **Optimal** — minimal false alarms (~16 false positives out of 1,929). |
| **PWF** (Power Failure) | 17 | 0.9444 | 1.0000 | 0.9714 | **Optimal** — 100% recall; `Power_Watts` cleanly captures the physical boundary. |
| **OSF** (Overstrain Failure) | 20 | 0.9500 | 0.9500 | 0.9500 | **Optimal** — `Overstrain_Index` isolates the torque-wear threshold accurately. |
| **HDF** (Heat Dissipation) | 21 | 0.8750 | 1.0000 | 0.9333 | **Strong** — 100% recall with only ~3 false positives; `Temp_Diff` is highly effective. |
| **TWF** (Tool Wear Failure) | 8 | 0.0833 | 0.1250 | 0.1000 | **Bottleneck** — only 1 of 8 true cases caught, with high false positives. |

---

## 3. Root Cause of the `TWF` Bottleneck
1. **Extreme Sample Scarcity:** Only **8 test instances** (~32 training instances) exist for `TWF`. Tree splits overfit or get drowned out by `No_Failure` samples in the same feature space.
2. **Stochastic Threshold Mechanics in the Dataset:** In the AI4I 2020 data generation rules, `TWF` does not trigger at a deterministic physical formula like `PWF` or `OSF`. Instead, `TWF` occurs randomly within a tool-wear window of **200–240 minutes** (or is replaced by a tool change), making `Tool wear` overlap heavily with normal operation (`No_Failure`).
3. **Aggressive Sample Weighting Side Effects:** Using `class_weight="balanced"` assigns massive weight to those ~32 `TWF` training rows, causing the model to over-predict `TWF` on normal high-wear machines (`Precision = 0.0833`).

---

## 4. Recommended Next Iterations to Boost Macro F1
* **Moderate Class Weights (Damped Weighting):** Replace full `"balanced"` sample weights with dampened weights (e.g., `weight ** 0.5`) or tune `max_delta_step=1` in `XGBClassifier` to stabilize gradient updates on ultra-rare classes (`TWF`) without triggering excessive false positives.
* **Threshold / Probability Calibration:** Instead of `argmax` (`model.predict()`), extract `predict_proba()` and tune class-specific probability thresholds on out-of-fold validation predictions to maximize Macro F1.
* **Tool-Wear Zone Features:** Add binary/distance features targeting the 200–240 minute wear window (e.g., `In_TWF_Window = (Tool_wear >= 200) & (Tool_wear <= 240)` and `Wear_Margin_200 = Tool_wear - 200`) to help shallow trees isolate the critical wear regime.